<a href="https://colab.research.google.com/github/rouyu0405/Song-Lover---Lyric-Generation-NLP-Project/blob/main/FINAL_CODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Song Lyric Generator - NLP Project**
This project has trained GPT2 to produce song lyrics based on different genres and song verse.


Connect to Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Import libraries

In [2]:
# install libs
!pip install -q evaluate datasets transformers accelerate bitsandbytes peft sentencepiece
!pip install -q nltk textstat syllapy
!pip install -q rouge_score
!pip install -U bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
import os
import re
import math
import random
from collections import Counter
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import torch

import nltk
nltk.download("punkt")
nltk.download("stopwords")

import textstat
import syllapy

from datasets import Dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)

from peft import get_peft_model, LoraConfig

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Device: cuda


Data Preprocessing

In [32]:
#import datasets
df_main = pd.read_csv("/content/drive/MyDrive/360/data/all_lyrics.csv")
df_extra = pd.read_csv("/content/drive/MyDrive/360/data/NewData.csv")
df = pd.concat([df_main, df_extra], ignore_index=True)

SECTION_KEYWORDS = {
    "intro": ["intro", "introduction"],
    "verse": ["verse"],
    "chorus": ["chorus", "hook"],
    "pre-chorus": ["pre-chorus", "prechorus", "pre chorus"],
    "post-chorus": ["post-chorus", "postchorus", "post chorus"],
    "outro": ["outro", "ending"]
}
SECTION_ORDER = ["intro", "verse", "pre-chorus", "chorus", "post-chorus", "outro"]

def canonical_section(word):
    w = word.lower().strip()
    for k, vals in SECTION_KEYWORDS.items():
        for v in vals:
            if v in w:
                return k
    return None

header_regex = re.compile(
    r'(?:\[*\b(?:intro|verse|chorus|pre-chorus|prechorus|pre chorus|post-chorus|postchorus|post chorus|outro)\b[^\]\n:]*\]*[:\s]*)',
    flags=re.IGNORECASE
)

def extract_sections(original_text):
    if not isinstance(original_text, str) or original_text.strip() == "":
        return []
    txt = original_text
    parts = []
    matches = list(header_regex.finditer(txt))
    if matches:
        for i, m in enumerate(matches):
            start = m.start()
            header = m.group().strip()
            body_start = m.end()
            body_end = matches[i+1].start() if i+1 < len(matches) else len(txt)
            body = txt[body_start:body_end].strip()
            label = canonical_section(header) or "verse"
            parts.append((label, body))
    else:
        chunks = [c.strip() for c in re.split(r'\n{2,}', txt) if c.strip()]
        for i, c in enumerate(chunks):
            label = "verse"
            if i == 0 and len(chunks) > 1 and len(c.split()) < 12:
                label = "intro"
            parts.append((label, c))
    cleaned_parts = []
    for label, body in parts:
        b = re.sub(r'\[.*?\]', '', body)
        b = re.sub(r'[^a-zA-Z0-9\s\.,?!\'\"\-\(\):;]', ' ', b)
        b = re.sub(r'[ \t]+', ' ', b)
        b = re.sub(r'\n{2,}', '\n', b)
        b = b.strip()
        if len(b.split()) >= 3:
            cleaned_parts.append((label, b.lower()))
    return cleaned_parts

def build_expanded(df_src):
    rows = []
    for _, row in df_src.iterrows():
        orig = row.get("lyrics", "")
        genre = str(row.get("genre", row.get("type", "unknown"))).lower().replace(" ", "_")
        sections = extract_sections(orig)
        if not sections:
            continue
        for sec_label, sec_text in sections:
            sec_text = sec_text.replace("\n", " <LB> ")
            rows.append({
                "genre": genre,
                "section": sec_label,
                "text": sec_text,
                "lyrics": f"<genre_{genre}> <section_{sec_label}> {sec_text}"
            })
    return pd.DataFrame(rows)

expanded_main = build_expanded(df_main)
expanded_extra = build_expanded(df_extra)
print("Main expanded rows:", len(expanded_main))
print("Extra expanded rows (test):", len(expanded_extra))

Main expanded rows: 28653
Extra expanded rows (test): 438


Dataset Stat

In [33]:
expanded_main["token_count"] = expanded_main["text"].apply(lambda x: len(x.split()))
expanded_main["line_count"] = expanded_main["text"].apply(lambda x: x.count("<LB>") + 1)
print(expanded_main["token_count"].describe())
print(expanded_main["line_count"].describe())

count    28653.000000
mean        98.107423
std        703.287733
min          3.000000
25%         24.000000
50%         44.000000
75%         79.000000
max      63411.000000
Name: token_count, dtype: float64
count    28653.000000
mean        10.084145
std         52.691955
min          1.000000
25%          3.000000
50%          5.000000
75%          9.000000
max       3110.000000
Name: line_count, dtype: float64


Train/Validation split

In [34]:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    expanded_main,
    test_size=0.10,
    random_state=SEED,
    stratify=expanded_main["genre"]
)
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(expanded_extra.reset_index(drop=True))

print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))

Train: 25787 Val: 2866 Test: 438


Tokenizer + special tokens

In [36]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # safe
unique_genres = sorted(expanded_main["genre"].unique().tolist())
section_types = ["intro", "verse", "pre-chorus", "chorus", "post-chorus", "outro"]
special_tokens = [f"<genre_{g}>" for g in unique_genres] + [f"<section_{s}>" for s in section_types] + ["<LB>"]
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})
print("Added special tokens:", len(special_tokens))

# --- Tokenization (with label masking for padding) ---
MAX_LEN = 128

def tokenize_batch(examples):
    enc = tokenizer(examples["lyrics"], truncation=True, max_length=MAX_LEN, padding="max_length")
    input_ids = enc["input_ids"]
    labels = []
    pad_id = tokenizer.pad_token_id
    for seq in input_ids:
        lab = [tok if tok != pad_id else -100 for tok in seq]  # mask padding for loss
        labels.append(lab)
    enc["labels"] = labels
    return enc

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tokenize_batch, batched=True, remove_columns=val_ds.column_names)
test_tok  = test_ds.map(tokenize_batch, batched=True, remove_columns=test_ds.column_names)

train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Added special tokens: 11


Map:   0%|          | 0/25787 [00:00<?, ? examples/s]

Map:   0%|          | 0/2866 [00:00<?, ? examples/s]

Map:   0%|          | 0/438 [00:00<?, ? examples/s]

Model

In [37]:
def build_base_model_fullfinetune(model_name="gpt2", device_map="auto"):
    # If GPU is limited, use bnb/4bit
    base = AutoModelForCausalLM.from_pretrained(model_name)
    base.resize_token_embeddings(len(tokenizer))
    return base

def build_model_with_lora(model_name="gpt2", lora_r=8, device_map="auto"):
    # bitsandbytes config - uses 4bit to reduce memory
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
        bnb_4bit_use_double_quant=True
    )
    base = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_cfg, device_map=device_map)
    base.resize_token_embeddings(len(tokenizer))
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=16,
        target_modules=["c_attn", "c_proj", "c_fc"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(base, lora_config)
    return model

In [38]:
# offensive word filter

BAD_WORDS = {
    "fuck", "shit", "bitch", "asshole", "dick", "pussy",
    "slut", "whore", "nigger", "nigga", "faggot", "retard",
    "cunt", "kill yourself", "rape", "motherfucker", "cum"
}

def clean_offensive_language(text, mask="****"):
    """
    Replaces offensive words with ****
    """
    if not isinstance(text, str):
        return text

    clean_text = text
    for word in BAD_WORDS:
        pattern = r"\b" + re.escape(word) + r"\b"
        clean_text = re.sub(pattern, mask, clean_text, flags=re.IGNORECASE)

    return clean_text


def contains_offensive_language(text):
    """
    Returns True if any bad word is found
    """
    text = text.lower()
    return any(bad_word in text for bad_word in BAD_WORDS)

In [39]:
def generate_from_prompt(model, prompt, max_new_tokens=80, **gen_kwargs):
    model.eval()
    tok = tokenizer(prompt, return_tensors="pt").to(model.device)
    # Safe defaults
    defaults = {"do_sample": True, "temperature": 0.7, "top_p": 0.92, "top_k": 50, "repetition_penalty": 1.1, "no_repeat_ngram_size": 3}
    for k,v in defaults.items():
        gen_kwargs.setdefault(k, v)
    with torch.no_grad():
        out = model.generate(
            **tok,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            **gen_kwargs
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    # Remove prompt portion and return the newly generated suffix if present
    generated = text[len(tokenizer.decode(tok["input_ids"][0], skip_special_tokens=True)):]

    if not generated.strip():
        generated = text.strip()

    # Replace tokens + cleanup
    generated = generated.replace("<LB>", "\n").strip()

    # APPLY BAD WORD FILTER
    if contains_offensive_language(generated):
        generated = clean_offensive_language(generated)

    return generated

In [40]:
# Grammar & plagiarism
# grammar_score: higher = easier -> treat as better grammar
def grammar_score(text):
    try:
        # Replace <LB> with newline for textstat
        t = text.replace("<LB>", "\n")
        score = textstat.flesch_reading_ease(t)
        # normalize to 0-1 roughly
        return max(0.0, min(1.0, score / 100.0))
    except Exception:
        return 0.5

# plagiarism_score: 1.0 = exact match -> bad
def plagiarism_score(generated, reference=None):
    toks_g = generated.split()
    if reference:
        toks_r = reference.split()
        common = len(set(toks_g).intersection(set(toks_r)))
        return common / max(1, len(toks_g))
    else:
        return 0.0

In [41]:
# syllable utilities for rhythm
def syllable_counts_for_lines(text):
    txt = text.replace("<LB>", "\n")
    lines = [l.strip() for l in txt.split("\n") if l.strip()]
    counts = []
    for l in lines:
        # count syllables word-by-word
        s = sum(syllapy.count(w) for w in re.findall(r"[A-Za-z']+", l))
        counts.append(s)
    return counts

def syllable_match_score(gen_text, ref_text):
    g_counts = syllable_counts_for_lines(gen_text)
    r_counts = syllable_counts_for_lines(ref_text)
    if not g_counts or not r_counts:
        return 0.0
    # compare mean syllable counts per line
    return 1.0 - (abs(np.mean(g_counts) - np.mean(r_counts)) / max(1.0, np.mean(r_counts)))

# distinct-n and repetition
def distinct_n(tokens, n=1):
    if len(tokens) == 0: return 0.0
    ngrams = list(zip(*[tokens[i:] for i in range(n)]))
    return len(set(ngrams)) / max(1, len(ngrams))

# perplexity helper
def compute_perplexity(eval_loss):
    return math.exp(eval_loss) if eval_loss is not None and eval_loss < 100 else float("inf")

In [42]:
from sentence_transformers import SentenceTransformer, util

# Lightweight sentence embedding model for line similarity
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

def evaluate_model(model, trainer_obj, val_tok, test_df, output_dir, gen_kwargs=None):
    # eval loss & perplexity on validation
    eval_results = trainer_obj.evaluate(eval_dataset=val_tok) if trainer_obj is not None else {}
    val_loss = eval_results.get("eval_loss", None)
    val_ppl = compute_perplexity(val_loss)

    gen_kwargs = gen_kwargs or {
        "temperature":0.7, "top_p":0.92, "top_k":60,
        "repetition_penalty":1.2, "no_repeat_ngram_size":3, "do_sample":True
    }

    hyps, refs = [], []
    grammar_scores, plagiarism_scores = [], []
    distinct1_list, distinct2_list = [], []
    syllable_matches = []
    line_counts, avg_line_lengths, rep_rates = [], [], []
    line_coherence_list = []

    for _, row in tqdm(test_df.head(80).iterrows(), total=80):  # limit down for faster training
        prompt = f"<genre_{row['genre']}> <section_{row['section']}> "
        gen = generate_from_prompt(model, prompt, max_new_tokens=120, **gen_kwargs)
        ref = row["text"].replace("<LB>", "\n").strip()

        hyps.append(gen)
        refs.append(ref)

        grammar_scores.append(grammar_score(gen))
        plagiarism_scores.append(plagiarism_score(gen, ref))

        toks = gen.split()
        distinct1_list.append(distinct_n(toks, 1))
        distinct2_list.append(distinct_n(toks, 2))

        lines = [l for l in gen.split("\n") if l.strip()]
        line_counts.append(len(lines))
        avg_line_lengths.append(np.mean([len(l.split()) for l in lines]) if lines else 0)

        # Repetition rate
        wc = Counter(toks)
        rep_rates.append(sum(c for c in wc.values() if c > 1) / max(1, len(toks)))

        # Line-to-line semantic coherence (cosine similarity)
        if len(lines) > 1:
            sims = []
            for i in range(len(lines)-1):
                emb1 = sbert_model.encode(lines[i], convert_to_tensor=True)
                emb2 = sbert_model.encode(lines[i+1], convert_to_tensor=True)
                sims.append(util.cos_sim(emb1, emb2).item())
            line_coherence_list.append(np.mean(sims))
        else:
            line_coherence_list.append(1.0)  # single line considered perfectly coherent

        syllable_matches.append(syllable_match_score(gen, ref))

    # corpus scores (BLEU/ROUGE)
    try:
        bleu_score = bleu.compute(predictions=[h.split() for h in hyps], references=[[r.split()] for r in refs])["bleu"]
    except Exception:
        bleu_score = 0.0
    try:
        rouge_score = rouge.compute(predictions=hyps, references=refs)["rouge1"]
    except Exception:
        rouge_score = 0.0

    out = {
        "val_loss": val_loss,
        "val_ppl": val_ppl,
        "bleu": float(bleu_score),
        "rouge1": float(rouge_score),
        "avg_grammar": float(np.mean(grammar_scores)),
        "avg_plagiarism": float(np.mean(plagiarism_scores)),
        "distinct_1": float(np.mean(distinct1_list)),
        "distinct_2": float(np.mean(distinct2_list)),
        "avg_line_count": float(np.mean(line_counts)),
        "avg_line_length": float(np.mean(avg_line_lengths)),
        "repetition_rate": float(np.mean(rep_rates)),
        "avg_line_coherence": float(np.mean(line_coherence_list)),  # new metric
        "avg_syllable_match": float(np.mean(syllable_matches)),
        "examples": list(zip(hyps[:5], refs[:5]))
    }

    # save generated pairs
    os.makedirs(output_dir, exist_ok=True)
    pd.DataFrame({"generated": hyps, "reference": refs}).to_csv(os.path.join(output_dir, "generated_vs_ref.csv"), index=False)
    print("Saved generated pairs to", os.path.join(output_dir, "generated_vs_ref.csv"))
    return out

Training Model

In [43]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [44]:
# 1) baseline_full = full fine-tune of small model

def train_and_eval_fullfinetune(output_dir="/content/fullfinetune", training_args=None):
    model = build_base_model_fullfinetune(model_name=model_name, device_map="auto")
    model.resize_token_embeddings(len(tokenizer))

    if training_args is None:
        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=4,
            max_steps=600,
            warmup_steps=50,
            learning_rate=5e-5,
            num_train_epochs=1,
            fp16=torch.cuda.is_available(),
            logging_steps=50,
            save_steps=200,
            save_total_limit=1,
            eval_strategy="steps",
            seed=SEED
        )
    trainer = Trainer(model=model, args=training_args, train_dataset=train_tok, eval_dataset=val_tok, data_collator=data_collator)
    print("Training baseline full-finetune")
    trainer.train()
    trainer.save_model(output_dir)
    results = evaluate_model(model, trainer, val_tok, test_ds.to_pandas(), output_dir)
    return model, trainer, results

In [45]:
# Baseline: full fine-tune (smaller LR)
baseline_model, baseline_trainer, baseline_results = train_and_eval_fullfinetune(output_dir="/content/fullfinetune")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Training baseline full-finetune


Step,Training Loss,Validation Loss
50,4.796000,4.235306
100,4.090500,3.812356
150,3.688800,3.385437
200,3.456700,3.211046
250,3.253100,3.164783
300,3.264200,3.127580
350,3.324500,3.106609
400,3.207400,3.075224
450,3.142100,3.058414
500,3.159100,3.045181


  0%|          | 0/80 [00:00<?, ?it/s]

Saved generated pairs to /content/fullfinetune/generated_vs_ref.csv


In [46]:
# 2) variant_lora = LoRA with changed hyperparameter `r`

def train_and_eval_lora(lora_r=8, output_dir="/content/lora", training_args=None):
    model = build_model_with_lora(model_name=model_name, lora_r=lora_r, device_map="auto")
    model.print_trainable_parameters()
    if training_args is None:
        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=4,
            max_steps=600,
            warmup_steps=50,
            learning_rate=2e-4,
            num_train_epochs=1,
            fp16=torch.cuda.is_available(),
            logging_steps=50,
            save_steps=200,
            save_total_limit=1,
            eval_strategy="steps",
            seed=SEED
        )
    trainer = Trainer(model=model, args=training_args, train_dataset=train_tok, eval_dataset=val_tok, data_collator=data_collator)
    print(f"Training LoRA r={lora_r}")
    trainer.train()
    trainer.save_model(output_dir)
    results = evaluate_model(model, trainer, val_tok, test_ds.to_pandas(), output_dir)
    return model, trainer, results

In [47]:
# Variant: changed one hyperparameter (LoRA r from 8 -> 16)
variant_model, variant_trainer, variant_results = train_and_eval_lora(lora_r=16, output_dir="/content/lora_r16")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


trainable params: 2,359,296 || all params: 126,807,552 || trainable%: 1.8605
Training LoRA r=16


Step,Training Loss,Validation Loss
50,5.258300,4.615144
100,4.505000,4.317243
150,4.353200,4.247948
200,4.306600,4.215873
250,4.226700,4.194914
300,4.286200,4.174294


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Step,Training Loss,Validation Loss
50,5.258300,4.615144
100,4.505000,4.317243
150,4.353200,4.247948
200,4.306600,4.215873
250,4.226700,4.194914
300,4.286200,4.174294
350,4.324400,4.164459
400,4.219500,4.155345
450,4.152000,4.146174
500,4.199500,4.139589


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


  0%|          | 0/80 [00:00<?, ?it/s]

Saved generated pairs to /content/lora_r16/generated_vs_ref.csv


Evaluation

In [48]:
def summarize_res(name, res):
    print(f"\n=== {name} ===")
    for k in ["val_loss","val_ppl","bleu","rouge1","distinct_1","distinct_2","avg_line_count","avg_line_length","repetition_rate","avg_syllable_match","avg_grammar","avg_plagiarism"]:
        print(f"{k}: {res.get(k)}")

summarize_res("Baseline (full-finetune)", baseline_results)
summarize_res("Variant (LoRA r=16)", variant_results)

# compare results table
comp_df = pd.DataFrame([
    {"model":"baseline_full", **{k: baseline_results.get(k) for k in baseline_results if k!="examples"}},
    {"model":"lora_r16", **{k: variant_results.get(k) for k in variant_results if k!="examples"}}
])
display(comp_df)


=== Baseline (full-finetune) ===
val_loss: 3.0357534885406494
val_ppl: 20.816657104266003
bleu: 0.0
rouge1: 0.1545790023885708
distinct_1: 0.9882625218718573
distinct_2: 1.0
avg_line_count: 1.1625
avg_line_length: 85.3
repetition_rate: 0.022472389726807405
avg_syllable_match: -12.214204420852482
avg_grammar: 0.789037830090895
avg_plagiarism: 0.08726076490489951

=== Variant (LoRA r=16) ===
val_loss: 4.135900020599365
val_ppl: 62.54585830671248
bleu: 0.0
rouge1: 0.16084610699135551
distinct_1: 0.9921447528336943
distinct_2: 1.0
avg_line_count: 1.3375
avg_line_length: 69.825
repetition_rate: 0.014829156198797872
avg_syllable_match: -9.56111197663817
avg_grammar: 0.8407253411625616
avg_plagiarism: 0.09406941143544974


,model,val_loss,val_ppl,bleu,rouge1,avg_grammar,avg_plagiarism,distinct_1,distinct_2,avg_line_count,avg_line_length,repetition_rate,avg_line_coherence,avg_syllable_match
0,baseline_full,3.035753,20.816657,0.0,0.154579,0.789038,0.087261,0.988263,1.0,1.1625,85.300,0.022472,0.870035,-12.214204
1,lora_r16,4.135900,62.545858,0.0,0.160846,0.840725,0.094069,0.992145,1.0,1.3375,69.825,0.014829,0.734873,-9.561112


# **Lyric Generation**

In [57]:
model_base = baseline_model
model_new = variant_model

Change this promt for different song topics

In [95]:
SECTION = "outro"
GENRE = "rock"
USER_PROMPT = "Ocean and the beach"

Run for baseline model results:

In [99]:
import torch

def build_prompt(user_prompt, section, genre):
    return f"""
You are a lyricist.

Write literal, coherent song lyrics.

Requirements for the lyrics:
- Each line must connect to the previous one in meaning
- Use clear cause-and-effect
- Avoid abstract or random jumps

Theme:
{user_prompt}

Lyric Section:
{section}

Genre:
{genre}

Lyrics:
"""

prompt = build_prompt(USER_PROMPT, SECTION, GENRE)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=False,
    truncation=False
).to(model_base.device)

with torch.no_grad():
    output = model_base.generate(
        **inputs,
        max_new_tokens=220,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


You are a lyricist.

Write literal, coherent song lyrics.

Requirements for the lyrics:
- Each line must connect to the previous one in meaning
- Use clear cause-and-effect
- Avoid abstract or random jumps

Theme:
Ocean and the beach

Lyric Section:
outro

Genre:
rock

Lyrics:
 (featuring) - "songs of my youth" by jonathan dane ryan watson  outrageous bullshit... shit that i say is true! so what? it's not like you're trying hard enough; but if there was anything else we could do better than this crap at your expense now please don't let me down with all these crazy things I'm telling ya tonight 'cause nobody cares about us anymore anyway just wait till tomorrow morning baby until then when they'll be up on their knees looking over our shoulder singing along lovingly as though nothing happened."by dj kriemann''instrumental music video from mike phillip' nyc s new album , available here . listen below without any warning before downloading because no matter how much time passes between 

Run for variant model results:

In [102]:
import torch

def build_prompt(user_prompt, section, genre):
    return f"""
You are a lyricist.

Write literal, coherent song lyrics.

Requirements for the lyrics:
- Each line must connect to the previous one in meaning
- Use clear cause-and-effect
- Avoid abstract or random jumps

Theme:
{user_prompt}

Lyric Section:
{section}

Genre:
{genre}

Lyrics:
"""

prompt = build_prompt(USER_PROMPT, SECTION, GENRE)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=False,
    truncation=False
).to(model_new.device)

with torch.no_grad():
    output = model_new.generate(
        **inputs,
        max_new_tokens=220,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


You are a lyricist.

Write literal, coherent song lyrics.

Requirements for the lyrics:
- Each line must connect to the previous one in meaning
- Use clear cause-and-effect
- Avoid abstract or random jumps

Theme:
Ocean and the beach

Lyric Section:
outro

Genre:
rock

Lyrics:
 'cause it's so easy...' (it is) my love of you , i'm your friend forever  (i am not alone)

 "all those words that say nothing about me" - this was how I felt when we sang these songs together on our first album with Lazy Soulz . The soundscape changes every few minutes at nighttime but as time passes by people will realize they're no longer friends anymore; their hearts have been broken from years long without them even knowing what happened because she had never met someone who could remember her name before now? It wasn't like being inside an old man walking down town living out his days through life while everyone else would think he'd just left home until after midnight instead! And yet all things were dif